# Lang Chain Quickstart Guild
https://docs.langchain.com/oss/python/langchain/quickstart

## 기본 에이전트를 구축하세요

먼저 질문에 답하고 도구를 호출할 수 있는 간단한 에이전트를 만드세요.  
이 예제의 에이전트는 선택한 언어 모델, 기본적인 날씨 기능(도구), 그리고 동작을 안내하는 간단한 프롬프트를 사용합니다.

In [3]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    reasoning_effort="none",
)

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=llm,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user", 
                "content": "What's the weather in San Francisco?"
            }
        ]
    }
)

print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'It’s always sunny in San Francisco!'}]


## 실제 환경에서 작동하는 에이전트를 구축하세요
다음 예제에서는 텍스트 파일에 대한 질문에 답할 수 있는 리서치 에이전트를 구축합니다. 이 과정에서 다음과 같은 개념들을 살펴보게 됩니다.

1. 상담원 동작 개선을 위한 상세 시스템 안내 메시지
2. 외부 데이터와 연동되는 도구를 개발하세요
3. 일관된 응답을 위한 모델 구성
4. 채팅과 유사한 상호작용을 위한 대화형 메모리
5. 내장 기능을 위한 딥 에이전트
6. 에이전트 테스트 중

### 1. 시스템 프롬프트를 정의합니다.
시스템 메시지는 상담원의 역할과 동작을 정의합니다. 구체적이고 실행 가능한 메시지를 작성하세요.

In [4]:
SYSTEM_PROMPT = """
You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file.
"""

## 2. 생성 도구
툴을 사용 하면 사용자가 정의한 함수를 호출하여 모델이 외부 시스템과 상호 작용할 수 있습니다.  
툴은 런타임 컨텍스트 에 의존할 수 있으며 에이전트 메모리 와도 상호 작용할 수 있습니다 .

In [5]:
import urllib.error
import urllib.request

from langchain.tools import tool


@tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL.
    """
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    text = raw.decode("utf-8", errors="replace")
    return text

## 3. 모델을 구성하세요
사용 사례에 맞는 매개변수로 언어 모델을 설정하세요 . 예를 들면 다음과 같습니다.

In [6]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gpt-5.6-luna",
    timeout=300,
    max_tokens=25000,
    reasoning_effort="none",
)

## 4. 메모리 추가
에이전트에 메모리를 추가하여 상호 작용 전반에 걸쳐 상태를 유지하세요. 이를 통해 에이전트는 이전 대화와 맥락을 기억할 수 있습니다.

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

## 5. 에이전트를 생성하고 실행합니다.
이제 모든 구성 요소를 사용하여 에이전트를 조립하고 실행하십시오.  
에이전트를 생성하는 데에는 LangChain 에이전트와 딥 에이전트라는 두 가지 프레임워크가 있습니다.  
LangChain 에이전트와 딥 에이전트 모두 도구, 메모리 등을 세밀하게 제어할 수 있도록 지원합니다.  
두 프레임워크의 주요 차이점은 딥 에이전트에는 계획 수립, 파일 시스템 도구, 하위 에이전트와 같이  
일반적으로 유용한 기능들이 기본적으로 내장되어 있다는 점입니다.

최소한의 설정으로 최대한의 기능을 원할 때는 딥 에이전트를 사용하고, 세밀한 제어가 필요할 때는 LangChain 에이전트를 선택하십시오.  
이 단계에서 두 가지를 비교하려면 deepagents패키지를 설치하세요.

In [8]:
from langchain.agents import create_agent
from deepagents import create_deep_agent

agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

deep_agent = create_deep_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

print("Running create_agent...", flush=True)
agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)
print("Running create_deep_agent...", flush=True)
deep_agent_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-da"}},
)
print("\ncreate_agent:")
print(agent_result["messages"][-1].content_blocks)
print("\ncreate_deep_agent:")
print(deep_agent_result["messages"][-1].content_blocks)

Running create_agent...
Running create_deep_agent...

create_agent:
[{'type': 'text', 'text': '{\n  "gatsby_line_count": null,\n  "first_daisy_line_number": null,\n  "synopsis": "Nick Carraway recounts the summer he spends among wealthy communities on Long Island, where he becomes involved with his enigmatic neighbor Jay Gatsby and Gatsby’s renewed pursuit of Daisy Buchanan, Nick’s cousin. Gatsby’s dream collides with Daisy’s marriage and the carelessness of the privileged, leading to betrayal, deaths, Gatsby’s murder, and Nick’s disillusioned departure from the East.",\n  "how_you_computed_counts": "I retrieved the complete Gutenberg text successfully, but the available tool returned the document as conversational text without a verifiable line-numbered representation or a machine-readable line-counting operation. Because I could not reliably count every line containing the exact substring `Gatsby` or establish the 1-based line number of the first line containing `Daisy`, both numeric

두 탭의 출력 결과를 살펴보면 LangChain 에이전트가 답변을 제공했지만 이는 추정치임을 알 수 있습니다.  
에이전트는 이 질문에 대한 정확한 답변을 제공할 수 있는 도구가 부족합니다.  
또한 프롬프트가 너무 길다는 오류 메시지가 나타날 수도 있습니다.  

반면, 심층 작용제는 다음과 같은 능력을 가지고 있습니다:
1. 내장된 write_todos도구를 사용하여 연구 작업을 세분화하고 접근 방식을 계획합니다.
2. fetch_text_from_url정보를 수집하는 도구를 호출하여 파일을 로드합니다.
3. grep파일 시스템 도구( 및 ) 를 사용하여 컨텍스트를 관리합니다read_file.
4. 복잡한 하위 작업을 전문화된 하위 에이전트에게 위임하기 위해 필요에 따라 하위 에이전트를 생성합니다.  

LangChain 에이전트의 경우, 유사한 수준의 서비스를 제공하려면 더 많은 기능을 구현해야 하며, 필요에 따라 구현 과정에서 이러한 기능을 맞춤 설정할 수 있습니다.